# Búsqueda local

Con la solución del greedy, buscamos ahora hacer una búsqueda local para intentar mejorar los resultados

In [ ]:
import pandas as pd 
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")


En la rama de **mas_cajas** los archivos csv pesados los tengo en las siguientes rutas:

In [ ]:
cajas_nuevas = pd.read_csv("../4r.cajas_nuevas.csv")
factibilidad = pd.read_csv("../Factibilidad/factibilidad_3mm.csv")

Vamos a cargar la mejor solución que tengamos (o la que querramos mejorar) para pasarla luego al algoritmo de búsqueda local:

In [ ]:
PATH_SOLUCION = 'Soluciones/solucion10-relocate&swap_3mm.csv'
solucion = pd.read_csv(f'{PATH_SOLUCION}')

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [14]:
def guardar_cajas_y_productos(grosor=3):
    
    cajas = {
    caja_id: Caja(caja_id=caja_id, dim_interior_ancho=ancho, dim_interior_largo=largo, dim_interior_alto=alto)
    for caja_id, ancho, largo, alto in zip(
        cajas_nuevas["caja_tipo_id"],
        cajas_nuevas["caja_interior_ancho"],
        cajas_nuevas["caja_interior_largo"],
        cajas_nuevas["caja_interior_alto"]
    )
}

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
    row.codigo_producto: Producto(
        codigo_producto=row.codigo_producto,
        cantidad_paquetes=row.cantidad_paquetes,
        peso_paquete=row.peso_neto_paquete,
        demanda_buenos_aires=row.volumen_producto_planta_buenos_aires,
        demanda_curitiba=row.volumen_producto_planta_curitiba,
        demanda_santiago=row.volumen_producto_planta_santiago,
        demanda_monterrey=row.volumen_producto_planta_monterrey,
        demanda_bakersfield=row.volumen_producto_planta_bakersfield,
        dim_producto_ancho=row.dim_producto_ancho,
        dim_producto_largo=row.dim_producto_largo,
        dim_producto_alto=row.dim_producto_alto
    )
    for row in prod_op_merge.itertuples(index=False)
}
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

#### **Reconstrucción de la solución inicial (Greedy)**

El csv exportado por `exportar_submmit` solo guarda las dimensiones *exteriores* de la caja asignada a cada producto, no el `caja_tipo_id`. Para poder operar con objetos `Caja` (y sus descuentos por volumen), reconstruimos el `caja_tipo_id` real restando el grosor a las dimensiones exteriores y buscando la caja correspondiente entre las cajas ya creadas.

In [15]:
grosor = 3
cajas, productos, cajas_asignables_por_producto = guardar_cajas_y_productos(grosor=grosor)

# Índice de cajas por sus dimensiones interiores (redondeadas), para poder
# encontrar el caja_tipo_id real a partir de las dimensiones exteriores del csv
indice_cajas_por_dim = {
    (round(c.dim_interior_ancho), round(c.dim_interior_largo), round(c.dim_interior_alto)): c
    for c in cajas.values()
}

solucion_inicial = Solucion(grosor, "Greedy 5 (maximizar utilización de pallet) - punto de partida")

for _, row in solucion.iterrows():
    producto = productos[row["codigo_producto"]]

    dim_ancho = round(row["caja_exterior_ancho"] - 2 * row["caja_grosor_mm"])
    dim_largo = round(row["caja_exterior_largo"] - 2 * row["caja_grosor_mm"])
    dim_alto = round(row["caja_exterior_alto"] - 2 * row["caja_grosor_mm"])

    caja = indice_cajas_por_dim[(dim_ancho, dim_largo, dim_alto)]

    asignacion = Asignacion(producto, caja)
    solucion_inicial.agregar_asignacion(asignacion)

solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 78
Costo packaging: 27296782.739999976
Costo flete: 161227950
Costo total: 188524732.73999998
Utilización de pallet promedio: 0.9579843026420239
Utilización de caja promedio: 0.9471952523608305
Ahorro costo total: 9.89813%


Lo que hace que tarde de correr tanto son las cantidad de combinaciones de cajas posibles por producto a considerar a la hora de evaluar el mejor movimiento posible:

In [18]:
import numpy as np

In [19]:
arr = np.array([len(cajas_asignables_por_producto.get(producto.codigo_producto, [])) for producto in list(productos.values())])
df =pd.DataFrame(arr)
df.describe()

,0
count,427.000000
mean,11564.316159
std,3831.308453
min,4669.000000
25%,8744.000000
50%,10763.000000
75%,14297.500000
max,29262.000000


#### **Cálculo del delta de costo de un movimiento**

El costo de packaging de una caja depende del volumen acumulado de **todos** los productos que la usan (por los descuentos por volumen). Por eso, mover un producto de una caja a otra no tiene un delta de costo aislado: afecta también el costo de los demás productos que ya comparten esa caja.

**Importante — versión no destructiva.** La primera versión de `calcular_delta_costo` simulaba el movimiento con `asignar_producto`/`revocar_producto` y después lo revertía. Eso mutaba el estado real de las cajas mientras evaluaba, y si la ejecución se interrumpía justo en el medio (como pasó), la caja quedaba con un estado inconsistente (algunas plantas actualizadas y otras no). La versión de acá abajo **no muta nada**: calcula el costo hipotético directamente con aritmética, usando `calcular_descuento_por_volumen` sobre las unidades que *tendría* la caja, sin llamar nunca a `asignar_producto`/`revocar_producto` durante la evaluación. Así, ninguna interrupción a mitad de camino puede dejar un estado corrupto — la única mutación real ocurre una sola vez por pasada, al aplicar el movimiento elegido.

El costo de flete sí es independiente del resto de los productos de la caja (depende solo de la demanda del producto y de `cantidad_cajas_por_pallet` de la caja), así que se calcula aparte con `Asignacion.cant_pallets_requeridas()`, sin necesidad de simular nada.

Nota: no volvemos a chequear factibilidad de dimensión/headspace/resistencia/utilización de pallet porque `cajas_asignables_por_producto` ya viene filtrado por esos criterios desde `5_factibilidad.ipynb`. Para el swap sí hay que chequear la factibilidad **cruzada** (que el producto A entre en la caja de B y viceversa), porque eso no está precomputado.

In [20]:
from Clases.caja import calcular_descuento_por_volumen

PLANTAS = ["buenos_aires", "curitiba", "santiago", "monterrey", "bakersfield"]


def costo_packaging_caja_hipotetico(caja, deltas_unidades):
    """
    Costo de packaging que tendría `caja` si sus unidades requeridas por
    planta cambiaran según `deltas_unidades` (dict planta -> delta, puede
    no incluir alguna planta si su delta es 0).
    Es puro cálculo: no lee ni modifica el estado real de la caja.
    """
    costo = 0
    for planta in PLANTAS:
        unidades_actuales = getattr(caja, f"unidades_{planta}_req")
        unidades_hipoteticas = unidades_actuales + deltas_unidades.get(planta, 0)
        descuento_hipotetico = calcular_descuento_por_volumen(unidades_hipoteticas)
        costo += unidades_hipoteticas * caja.costo_unitario * (1 + descuento_hipotetico)
    return costo


def calcular_delta_costo(producto, caja_actual, caja_nueva):
    """
    Delta de costo total (packaging + flete) si se reasignara `producto`
    de `caja_actual` a `caja_nueva`. No muta nada.
    """
    demanda = {planta: getattr(producto, f"demanda_{planta}") for planta in PLANTAS}

    costo_antes = (costo_packaging_caja_hipotetico(caja_actual, {})
                   + costo_packaging_caja_hipotetico(caja_nueva, {}))

    costo_despues = (costo_packaging_caja_hipotetico(caja_actual, {p: -d for p, d in demanda.items()})
                      + costo_packaging_caja_hipotetico(caja_nueva, demanda))

    delta_packaging = costo_despues - costo_antes

    # --- Costo de flete (independiente de otros productos de la caja) ---
    pallets_actual = Asignacion(producto, caja_actual).cant_pallets_requeridas()
    pallets_nueva = Asignacion(producto, caja_nueva).cant_pallets_requeridas()
    delta_flete = 150 * (pallets_nueva - pallets_actual)

    return delta_packaging + delta_flete


def calcular_delta_costo_swap(producto_a, caja_a, producto_b, caja_b):
    """
    Delta de costo total si se intercambiaran las cajas entre `producto_a`
    (actualmente en `caja_a`) y `producto_b` (actualmente en `caja_b`).
    Asume caja_a != caja_b. No muta nada.
    """
    demanda_a = {planta: getattr(producto_a, f"demanda_{planta}") for planta in PLANTAS}
    demanda_b = {planta: getattr(producto_b, f"demanda_{planta}") for planta in PLANTAS}

    costo_antes = (costo_packaging_caja_hipotetico(caja_a, {})
                   + costo_packaging_caja_hipotetico(caja_b, {}))

    delta_caja_a = {p: demanda_b[p] - demanda_a[p] for p in PLANTAS}  # sale A, entra B
    delta_caja_b = {p: demanda_a[p] - demanda_b[p] for p in PLANTAS}  # sale B, entra A

    costo_despues = (costo_packaging_caja_hipotetico(caja_a, delta_caja_a)
                       + costo_packaging_caja_hipotetico(caja_b, delta_caja_b))

    delta_packaging = costo_despues - costo_antes

    # --- Costo de flete: A pasa a caja_b, B pasa a caja_a ---
    pallets_a_antes = Asignacion(producto_a, caja_a).cant_pallets_requeridas()
    pallets_a_despues = Asignacion(producto_a, caja_b).cant_pallets_requeridas()
    pallets_b_antes = Asignacion(producto_b, caja_b).cant_pallets_requeridas()
    pallets_b_despues = Asignacion(producto_b, caja_a).cant_pallets_requeridas()

    delta_flete = 150 * ((pallets_a_despues - pallets_a_antes) + (pallets_b_despues - pallets_b_antes))

    return delta_packaging + delta_flete

#### **Búsqueda local: mejor mejora por pasada, con reasignación individual + swap**

En cada pasada se evalúan dos vecindarios y se aplica el mejor movimiento encontrado entre los dos (steepest descent global):

- **Reasignación individual**: mover un producto a otra caja factible (vecindario original).
- **Swap**: intercambiar las cajas entre dos productos, cuando cada uno es factible en la caja del otro. Sirve para los casos en que mover A o mover B por separado no mejora el costo (porque la caja que pierde volumen sube de tramo de descuento y compensa la ganancia), pero el intercambio neto sí conviene.

Se repite hasta que no se encuentra ninguna mejora en ninguno de los dos vecindarios (óptimo local) o se alcanza `max_iteraciones`.

Con 427 productos, evaluar todos los pares para el swap son ~91.000 combinaciones por pasada — no hace falta acotarlo. La reasignación individual sigue siendo la parte más pesada por la cantidad de cajas candidatas por producto (hasta varios miles), así que ahí sigue disponible `max_candidatos_por_producto` si hace falta.

**Sobre la interrupción manual:** como `calcular_delta_costo` y `calcular_delta_costo_swap` ya no mutan nada durante la evaluación (ver celda anterior), la única mutación real por pasada ocurre al aplicar el movimiento elegido, una sola vez. Esa ventana es mucho más chica que antes, pero sigue envuelta en `try/except KeyboardInterrupt` por las dudas.

In [21]:
import random

def busqueda_local(solucion, cajas, cajas_asignables_por_producto, max_iteraciones=50,
                    incluir_swap=True, max_candidatos_por_producto=None, verbose=True, semilla=42):
    """
    Búsqueda local por 'mejor mejora' (steepest descent) que en cada pasada
    evalúa dos tipos de movimiento (reasignación individual y swap entre
    pares de productos) y aplica el mejor de los dos encontrado. Modifica
    `solucion` in-place y devuelve un DataFrame con el historial de
    movimientos aplicados.

    Si se interrumpe manualmente (KeyboardInterrupt), se devuelve igual el
    historial parcial armado hasta ese momento.
    """
    random.seed(semilla)
    historial = []
    costo_actual = solucion.costo_total()
    UMBRAL_MEJORA = 1e-6  # tolerancia para evitar ciclos por ruido de punto flotante
    iteracion = 0

    # Versión en set de las cajas candidatas por producto, para poder chequear
    # factibilidad cruzada en el swap en O(1) en vez de recorrer la lista
    candidatos_set = {
        codigo: set(lista) for codigo, lista in cajas_asignables_por_producto.items()
    }

    try:
        for iteracion in range(1, max_iteraciones + 1):
            mejor_delta = -UMBRAL_MEJORA
            mejor_movimiento = None  # ("mover", asignacion, caja_nueva) o ("swap", asig_a, asig_b)

            # --- Vecindario 1: reasignación individual ---
            for asignacion in solucion.asignaciones:
                producto = asignacion.producto
                caja_actual = asignacion.caja
                candidatos = cajas_asignables_por_producto.get(producto.codigo_producto, [])

                if max_candidatos_por_producto is not None and len(candidatos) > max_candidatos_por_producto:
                    candidatos = random.sample(candidatos, max_candidatos_por_producto)

                for caja_id_candidata in candidatos:
                    if caja_id_candidata == caja_actual.caja_id:
                        continue

                    caja_candidata = cajas[caja_id_candidata]
                    delta = calcular_delta_costo(producto, caja_actual, caja_candidata)

                    if delta < mejor_delta:
                        mejor_delta = delta
                        mejor_movimiento = ("mover", asignacion, caja_candidata)

            # --- Vecindario 2: swap entre pares de productos ---
            if incluir_swap:
                asignaciones = solucion.asignaciones
                n = len(asignaciones)
                for i in range(n):
                    asig_a = asignaciones[i]
                    producto_a = asig_a.producto
                    caja_a = asig_a.caja

                    for j in range(i + 1, n):
                        asig_b = asignaciones[j]
                        producto_b = asig_b.producto
                        caja_b = asig_b.caja

                        if caja_a is caja_b:
                            continue  # ya comparten caja, el swap no tiene efecto

                        # Factibilidad cruzada: A tiene que entrar en la caja de B y viceversa
                        if caja_b.caja_id not in candidatos_set.get(producto_a.codigo_producto, set()):
                            continue
                        if caja_a.caja_id not in candidatos_set.get(producto_b.codigo_producto, set()):
                            continue

                        delta = calcular_delta_costo_swap(producto_a, caja_a, producto_b, caja_b)

                        if delta < mejor_delta:
                            mejor_delta = delta
                            mejor_movimiento = ("swap", asig_a, asig_b)

            if mejor_movimiento is None:
                if verbose:
                    print(f"Iteración {iteracion}: no se encontraron mejoras. Óptimo local alcanzado.")
                break

            tipo_movimiento = mejor_movimiento[0]

            if tipo_movimiento == "mover":
                _, asignacion, caja_nueva = mejor_movimiento
                producto = asignacion.producto
                caja_vieja = asignacion.caja

                caja_vieja.revocar_producto(producto)
                caja_nueva.asignar_producto(producto)
                asignacion.caja = caja_nueva

                # Actualizamos el tracking de tipos de caja utilizados en la solución
                if caja_nueva not in solucion.tipos_cajas_utilizados:
                    solucion.tipos_cajas_utilizados.append(caja_nueva)
                    solucion.cantidad_tipos_cajas += 1
                if len(caja_vieja.productos_asignados) == 0 and caja_vieja in solucion.tipos_cajas_utilizados:
                    solucion.tipos_cajas_utilizados.remove(caja_vieja)
                    solucion.cantidad_tipos_cajas -= 1

                descripcion = f"{producto.codigo_producto}: {caja_vieja.caja_id} -> {caja_nueva.caja_id}"
                registro = {
                    "iteracion": iteracion,
                    "tipo_movimiento": "mover",
                    "codigo_producto": producto.codigo_producto,
                    "caja_anterior": caja_vieja.caja_id,
                    "caja_nueva": caja_nueva.caja_id,
                    "codigo_producto_2": None,
                    "delta_costo": mejor_delta,
                }

            else:  # swap
                _, asig_a, asig_b = mejor_movimiento
                producto_a, caja_a = asig_a.producto, asig_a.caja
                producto_b, caja_b = asig_b.producto, asig_b.caja

                caja_a.revocar_producto(producto_a)
                caja_b.revocar_producto(producto_b)
                caja_b.asignar_producto(producto_a)
                caja_a.asignar_producto(producto_b)

                asig_a.caja = caja_b
                asig_b.caja = caja_a

                # Ninguna de las dos cajas puede quedar vacía en un swap
                # (cada una sigue teniendo al menos el producto que recibió),
                # así que no hace falta tocar tipos_cajas_utilizados.

                descripcion = (f"swap {producto_a.codigo_producto} <-> {producto_b.codigo_producto} "
                               f"entre {caja_a.caja_id} y {caja_b.caja_id}")
                registro = {
                    "iteracion": iteracion,
                    "tipo_movimiento": "swap",
                    "codigo_producto": producto_a.codigo_producto,
                    "caja_anterior": caja_a.caja_id,
                    "caja_nueva": caja_b.caja_id,
                    "codigo_producto_2": producto_b.codigo_producto,
                    "delta_costo": mejor_delta,
                }

            costo_actual += mejor_delta
            registro["costo_total_estimado"] = costo_actual
            historial.append(registro)

            if verbose:
                print(f"Iteración {iteracion}: {descripcion} "
                      f"(Δ={mejor_delta:,.2f} | costo total ≈ {costo_actual:,.2f})")

    except KeyboardInterrupt:
        if verbose:
            print(f"\nInterrumpido manualmente en la iteración {iteracion}. "
                  f"Se conservan los {len(historial)} movimientos ya aplicados.")

    return pd.DataFrame(historial)

#### **Ejecutamos la búsqueda local**

In [40]:
historial_busqueda_local = busqueda_local(
    solucion_inicial,
    cajas,
    cajas_asignables_por_producto,
    max_iteraciones=30,                # subir si todavía encuentra mejoras al llegar al límite
    incluir_swap=True,                 # poner False para volver a solo reasignación individual
    max_candidatos_por_producto=None,  # poner un número (ej. 200) si es muy lento
    verbose=True
)

historial_busqueda_local

Iteración 1: BR0196: CAJ2492044 -> CAJ2170604 (Δ=-3,721.56 | costo total ≈ 188,199,535.80)
Iteración 2: BR0051: CAJ1795068 -> CAJ1730780 (Δ=-13.32 | costo total ≈ 188,199,522.48)
Iteración 3: BR0086: CAJ1811140 -> CAJ1730780 (Δ=-7.02 | costo total ≈ 188,199,515.46)
Iteración 4: BR0089: CAJ1939716 -> CAJ1923644 (Δ=-4.74 | costo total ≈ 188,199,510.72)
Iteración 5: BR0301: CAJ2196868 -> CAJ2132580 (Δ=-3.24 | costo total ≈ 188,199,507.48)
Iteración 6: BR0057: CAJ1762924 -> CAJ1730780 (Δ=-2.10 | costo total ≈ 188,199,505.38)
Iteración 7: no se encontraron mejoras. Óptimo local alcanzado.


,iteracion,tipo_movimiento,codigo_producto,caja_anterior,caja_nueva,codigo_producto_2,delta_costo,costo_total_estimado
0,1,mover,BR0196,CAJ2492044,CAJ2170604,None,-3721.56,1.881995e+08
1,2,mover,BR0051,CAJ1795068,CAJ1730780,None,-13.32,1.881995e+08
2,3,mover,BR0086,CAJ1811140,CAJ1730780,None,-7.02,1.881995e+08
3,4,mover,BR0089,CAJ1939716,CAJ1923644,None,-4.74,1.881995e+08
4,5,mover,BR0301,CAJ2196868,CAJ2132580,None,-3.24,1.881995e+08
5,6,mover,BR0057,CAJ1762924,CAJ1730780,None,-2.10,1.881995e+08


#### **Comparación: Greedy vs. Búsqueda local**

In [41]:
solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 58
Costo packaging: 27027055.379999984
Costo flete: 161172450
Costo total: 188199505.38
Utilización de pallet promedio: 0.9538831132280713
Utilización de caja promedio: 0.9517688770636294
Ahorro costo total: 10.05357%


Exportamos la solución mejorada al mismo formato usado por las soluciones greedy:

In [42]:
solucion_inicial.exportar_submmit(nombre_csv="10-relocate&swap_3mm")